<a href="https://colab.research.google.com/github/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_5/lessons/lesson_49_docker/note_lesson_49_docker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🐳 Урок 49 — Docker: `news_hub` в образі

Агрегатор `news_hub` уже збирає новини, кешує їх у Redis, аналізує моделлю й має Telegram-бота. Сьогодні він стає **образом Docker**: один `Dockerfile` — і на будь-якій машині той самий застосунок з тими самими бібліотеками.

Більшість того, що Docker робить «усередині», можна відтворити чистим Python і Linux: розібрати `Dockerfile`, застосувати правила `.dockerignore`, порахувати кеш шарів, знайти секрет, надіслати процесу `SIGTERM`. Так і робимо — це працює і в Colab, де Docker немає. А в розділі 8 ті самі речі перевіряємо **справжнім Docker**, якщо він у тебе встановлений.

| Розділ | Що всередині | Приклади | Задача |
|---|---|---|---|
| 0 | проєкт `news_hub` і перевірка, чи є Docker | 0.1–0.2 | |
| 1 | `Dockerfile` як дані: інструкції, продовження рядків, exec- і shell-форма, `ENV`, шари | 1.1–1.5 | |
| 2 | `.dockerignore`: шаблони, винятки `!`, контекст збірки | 2.1–2.6 | **Задача 1** |
| 3 | кеш шарів: чому порядок інструкцій важливий | 3.1–3.4 | |
| 4 | секрети: `ENV` в метаданих, шар з `.env`, `--env-file` і лапки | 4.1–4.5 | **Задача 2** |
| 5 | сигнали: чому `docker stop` чекає 10 секунд | 5.1–5.6 | |
| 6 | здоров'я і порти: `/health`, `/health/ready`, `HEALTHCHECK`, `-p` | 6.1–6.3 | |
| 7 | знайди помилку: лінтер `Dockerfile` | | **Задача 3** |
| 8 | справжній Docker (якщо встановлений): контекст, кеш, секрети, зупинка, користувач, образ `news_hub` | 8.1–8.8 | |

**Як працювати**

- Виконуй клітинки **зверху вниз**. Клітинка з визначенням (`def`, константи, текст `Dockerfile`) лише створює — **запуск завжди в наступній клітинці**.
- Перед **🔮 Прогнозом** спершу відповідай сам, потім розгортай відповідь.
- Усе, крім розділу 8, — стандартна бібліотека Python і bash: нічого встановлювати не треба.
- Теорія, діаграми й реальні вимірювання — у книзі: [Урок 49](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m5/lesson_49/). Процеси, сигнали й права — у [бонус-уроці Linux](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m5/bonus_linux/).

---
## 0. Проєкт

### Приклад 0.1. Завантажити `news_hub` уроку 49

Поруч з ноутбуком має лежати папка `news_hub`. У Colab її немає — клітинка завантажить з GitHub лише папку цього уроку (`sparse-checkout`). Файли проєкту ми лише **читаємо**; усе, що створюємо самі, кладемо в окрему папку `docker_lab/`.

In [ ]:
import os
import subprocess
from pathlib import Path

LESSON = "module_5/lessons/lesson_49_docker"
REPO = "https://github.com/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026.git"


def find_project() -> Path:
    """Папка news_hub поруч з ноутбуком; якщо її немає (Colab) — завантажити лише цей урок з GitHub."""
    if Path("news_hub/Dockerfile").exists():
        return Path("news_hub").resolve()
    if not Path("course_repo").is_dir():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", REPO, "course_repo"],
                       check=True)
        subprocess.run(["git", "-C", "course_repo", "sparse-checkout", "set", LESSON], check=True)
    return (Path("course_repo") / LESSON / "news_hub").resolve()

In [ ]:
PROJECT = find_project()
LAB = Path("docker_lab").resolve()            # наші тимчасові файли — не в папці проєкту
LAB.mkdir(exist_ok=True)

DOCKERFILE = (PROJECT / "Dockerfile").read_text(encoding="utf-8")
DOCKERIGNORE = (PROJECT / ".dockerignore").read_text(encoding="utf-8").splitlines()
print("проєкт:", PROJECT.name, "| файлів у Dockerfile:", len(DOCKERFILE.splitlines()), "рядків")
print(DOCKERFILE)

Прочитай `Dockerfile` згори донизу — це рецепт: базовий образ `python:3.12-slim`, змінні середовища, залежності окремим шаром, користувач `app`, код, `HEALTHCHECK` і команда запуску. Далі розбиратимемо його по рядку.

### Приклад 0.2. Чи є тут Docker?

У Colab Docker немає: там немає демона, який запускає контейнери. Перевіримо так само, як це робитимуть клітинки розділу 8: чи є програма `docker` і чи відповідає демон (`docker info`).

In [ ]:
import shutil


def docker_ready() -> bool:
    """Є команда docker і демон відповідає. Без цього розділ 8 лише друкує пояснення."""
    if not shutil.which("docker"):
        return False
    try:
        return subprocess.run(["docker", "info"], capture_output=True, timeout=20).returncode == 0
    except (OSError, subprocess.TimeoutExpired):
        return False

In [ ]:
DOCKER = docker_ready()
print("Docker доступний:", DOCKER)
if not DOCKER:
    print("Docker немає (як у Colab) — розділи 1–7 працюють без нього, розділ 8 покаже вивід справжнього запуску.")

---
## 1. `Dockerfile` як дані

Кожен рядок `Dockerfile` — **інструкція** (`FROM`, `COPY`, `RUN`, …) і її аргументи. Рядок, що закінчується на `\`, продовжується на наступному; `#` на початку рядка — коментар. Docker сам розбирає файл саме так, тож напишемо такий розбір — він знадобиться в усіх наступних розділах.

### Приклад 1.1. `parse_dockerfile`: список інструкцій

Задача: перетворити текст на список пар `(ІНСТРУКЦІЯ, аргументи)`. Що робимо з рядком:

- порожній чи коментар — пропускаємо (коментар можна писати й усередині продовження);
- закінчується на `\` — відкладаємо в `buffer` і чекаємо продовження;
- інакше — склеюємо з `buffer` і ділимо на назву й аргументи за першим пробілом.

In [ ]:
def parse_dockerfile(text: str) -> list[tuple[str, str]]:
    """Інструкції Dockerfile: (НАЗВА, аргументи); продовження рядків склеєні через пробіл, коментарі пропущено."""
    instructions: list[tuple[str, str]] = []
    buffer = ""
    for line in text.splitlines():
        stripped = line.strip()
        if not stripped or stripped.startswith("#"):
            continue
        if stripped.endswith("\\"):
            buffer += stripped[:-1].strip() + " "
            continue
        full = buffer + stripped
        buffer = ""
        name, _, args = full.partition(" ")
        instructions.append((name.upper(), args.strip()))
    return instructions

In [ ]:
steps = parse_dockerfile(DOCKERFILE)
for number, (name, args) in enumerate(steps, 1):
    print(f"{number:2}. {name:<11} {args[:80]}")

names = [name for name, _ in steps]
assert names[0] == "FROM" and names[-1] == "CMD"
assert names.index("COPY") < names.index("RUN")                       # спершу requirements.txt, потім pip
assert ("COPY", ". .") in steps and steps.index(("COPY", ". .")) > names.index("RUN")

12 інструкцій. Порядок уже розповідає про задум: `COPY requirements.txt .` і `RUN pip install` стоять **до** `COPY . .` — це про кеш (розділ 3); `USER app` — **після** коду, щоб код належав root (розділ 8.6).

### Приклад 1.2. Продовження рядків і коментарі

Перевіримо розбір на маленькому тексті, де є все складне одразу: продовження `\`, коментар усередині продовження і інструкція малими літерами (Docker приймає й так).

In [ ]:
TRICKY = r"""
# коментар на початку
run apt-get update \
    # коментар усередині продовження — Docker його пропускає
    && apt-get install -y --no-install-recommends curl \
    && rm -rf /var/lib/apt/lists/*

ENV A=1
"""

In [ ]:
for name, args in parse_dockerfile(TRICKY):
    print(f"{name:<4} | {args}")
assert parse_dockerfile(TRICKY)[0][0] == "RUN"
assert "&& apt-get install" in parse_dockerfile(TRICKY)[0][1]

Три фізичні рядки `RUN` стали однією інструкцією, коментар між ними зник, `run` став `RUN`. Так само склеївся `HEALTHCHECK` проєкту з двох рядків.

### Приклад 1.3. exec-форма і shell-форма `CMD`

`CMD` (і `RUN`, `ENTRYPOINT`) пишуть двома способами:

- **exec-форма** — JSON-масив `["uvicorn", "news_hub.api:app", …]`: Docker запускає саме цю програму;
- **shell-форма** — звичайний рядок: Docker загортає його в `["/bin/sh", "-c", "рядок"]`, і першим процесом стає оболонка.

Функція повертає, яку саме команду запустить Docker.

In [ ]:
import json


def command_of(args: str) -> list[str]:
    """Що Docker справді запустить для аргументів CMD / RUN / ENTRYPOINT."""
    if args.startswith("["):
        return json.loads(args)                      # exec-форма: як є
    return ["/bin/sh", "-c", args]                   # shell-форма: через оболонку

In [ ]:
cmd_args = dict(steps)["CMD"]
print("news_hub:   ", command_of(cmd_args))
print("shell-форма:", command_of("uvicorn news_hub.api:app --host 0.0.0.0 --port 8000"))
assert command_of(cmd_args)[0] == "uvicorn"

У `news_hub` першим процесом буде сам `uvicorn`. У shell-формі — `/bin/sh`, а `uvicorn` — його дитиною. Різниця здається косметичною, але саме через неї `docker stop` чекає 10 секунд (розділ 5).

### Приклад 1.4. Які змінні середовища матиме контейнер

`ENV` буває у двох формах: `ENV A=1 B=2` і старій `ENV A 1`. Кожна наступна `ENV` доповнює попередні. Зберемо підсумкове середовище образу — те, що покаже `docker inspect`.

In [ ]:
import shlex


def parse_env(args: str) -> dict[str, str]:
    """Аргументи ENV / ARG → словник; підтримує `A=1 B=2` і стару форму `A 1`."""
    parts = shlex.split(args)
    if "=" not in parts[0]:                          # ENV NAME value з пробілами
        return {parts[0]: " ".join(parts[1:])}
    return dict(part.split("=", 1) for part in parts)


def image_env(steps: list[tuple[str, str]]) -> dict[str, str]:
    env: dict[str, str] = {}
    for name, args in steps:
        if name == "ENV":
            env.update(parse_env(args))
    return env

In [ ]:
for key, value in image_env(steps).items():
    print(f"{key:<30} = {value}")
print(parse_env("BOT_TOKEN 123:ABC"))
assert image_env(steps)["DATABASE_URL"].endswith("/data/news_hub.db")

Чотири змінні з першої `ENV` і `DATABASE_URL` з другої. Секретів тут немає — і так має бути: усе з `ENV` видно кожному, хто має образ (розділ 4).

### Приклад 1.5. Які інструкції створюють шари файлів

Образ — стос **шарів**. Файли змінюють лише `RUN`, `COPY` і `ADD`; решта (`ENV`, `USER`, `CMD`, `EXPOSE`, …) записує лише **метадані** образу й нічого не важить.

In [ ]:
FILE_LAYERS = {"RUN", "COPY", "ADD"}


def layer_kinds(steps: list[tuple[str, str]]) -> dict[str, list[str]]:
    kinds: dict[str, list[str]] = {"шар файлів": [], "метадані": []}
    for name, args in steps[1:]:                     # FROM — це базовий образ, не новий шар
        kinds["шар файлів" if name in FILE_LAYERS else "метадані"].append(f"{name} {args[:30]}")
    return kinds

In [ ]:
for kind, items in layer_kinds(steps).items():
    print(f"{kind} ({len(items)}):")
    for item in items:
        print("   ", item)

Лише 4 інструкції змінюють файли: `COPY requirements.txt`, `RUN pip install` (найбільший шар — 192 МБ), `RUN useradd …` і `COPY . .` (код — 336 кБ). `WORKDIR` хіба що створює порожню папку, решта — метадані. Тому «додати ще одну `ENV`» образ не роздуває, а кожен зайвий файл у `COPY . .` — так.

---
## 2. `.dockerignore`: що потрапить в образ

`docker build .` спершу пакує **контекст** — усю папку, крім того, що в `.dockerignore`. Правила Docker:

- шлях рахується **від кореня контексту**; `/` на початку й у кінці шаблону не важить;
- `*` — будь-що в межах однієї папки, `**` — будь-яка кількість папок;
- шаблон, що збігся з папкою, виключає все всередині;
- `!шаблон` — виняток: повертає файл назад;
- перевіряються **всі** рядки згори донизу, перемагає **останній**, що збігся.

### Приклад 2.1. Шаблон → регулярний вираз

`pattern_to_regex` перекладає шаблон `.dockerignore` у регулярний вираз: `**/` → «будь-які папки або жодної», `*` → «будь-що, крім `/`», а в кінці — «або щось усередині цієї папки».

In [ ]:
import re


def pattern_to_regex(pattern: str) -> re.Pattern[str]:
    pattern = pattern.strip("/")
    out, i = "", 0
    while i < len(pattern):
        if pattern.startswith("**/", i):
            out, i = out + "(?:.*/)?", i + 3
        elif pattern.startswith("**", i):
            out, i = out + ".*", i + 2
        elif pattern[i] == "*":
            out, i = out + "[^/]*", i + 1
        elif pattern[i] == "?":
            out, i = out + "[^/]", i + 1
        elif pattern[i] == "[":
            end = pattern.index("]", i)
            out, i = out + pattern[i:end + 1], end + 1
        else:
            out, i = out + re.escape(pattern[i]), i + 1
    return re.compile(out + "(?:/.*)?$")           # збіг з папкою — це й усе в ній

In [ ]:
for pattern, path in [("**/__pycache__", "news_hub/bot/__pycache__/api.pyc"), ("**/__pycache__", "__pycache__/x.pyc"),
                      ("*.db", "news_hub.db"), ("*.db", "data/news_hub.db"), ("**/*.db", "data/news_hub.db"),
                      ("tests/", "tests/unit/test_models.py"), (".env.*", ".env.production")]:
    regex = pattern_to_regex(pattern)
    print(f"{pattern:<16} {path:<36} → {bool(regex.match(path))!s:<5}  {regex.pattern}")

Зверни увагу на пару `*.db`: `news_hub.db` у корені виключено, а `data/news_hub.db` — **ні**: `*` не переходить через `/`. Тому в `.dockerignore` проєкту — `**/*.db`.

### Приклад 2.2. `excluded`: чи виключить Docker файл

Проходимо всі рядки `.dockerignore` згори донизу; кожен, що збігся, переписує результат (`!` — на «повернути»). Перемагає останній.

In [ ]:
def excluded(path: str, patterns: list[str]) -> bool:
    """Чи виключить Docker файл path (шлях від кореня контексту, через /)."""
    result = False
    for raw in patterns:
        line = raw.strip()
        if not line or line.startswith("#"):
            continue
        negate = line.startswith("!")
        if pattern_to_regex(line[1:] if negate else line).match(path):
            result = not negate
    return result


CASES = {
    ".env": True, ".env.production": True, ".env.example": False,
    ".venv/lib/python3.12/site-packages/fastapi/__init__.py": True,
    "news_hub/__pycache__/api.cpython-313.pyc": True, "news_hub/api.py": False,
    "tests/unit/test_models.py": True, "news_hub.db": True, "data/rbc_news_snapshot.json": False,
    "migrations/versions/0001_news_table.py": False, "requirements.txt": False, "requirements-dev.txt": True,
}

In [ ]:
for path, expected in CASES.items():
    got = excluded(path, DOCKERIGNORE)
    print(f"{'виключено' if got else 'в образ  '}  {path}")
    assert got == expected, path

В образ ідуть код, міграції, знімок новин і `requirements.txt`. Не йдуть секрети (`.env`, `.env.production`), локальне оточення (`.venv`), кеші, тести, база й залежності розробки. `.env.example` повернуто винятком — у ньому немає секретів, лише назви змінних.

### Приклад 2.3. Шаблон без `**/`

🔮 **Прогноз:** старий шаблон `__pycache__/` (без `**/`). Чи виключить він `news_hub/__pycache__/api.cpython-313.pyc`?

In [ ]:
old_rule = excluded("news_hub/__pycache__/api.cpython-313.pyc", ["__pycache__/"])
root_cache = excluded("__pycache__/x.pyc", ["__pycache__/"])
print("__pycache__/ виключає news_hub/__pycache__/…:", old_rule)
print("__pycache__/ виключає ./__pycache__/…:       ", root_cache)
assert old_rule is False and root_cache is True

<details>
<summary>Відповідь</summary>

Ні. Шаблон рахується від кореня: `__pycache__/` — це лише `./__pycache__`. Кеші в `news_hub/` і `news_hub/bot/` доїхали б в образ — і шар коду інвалідувався б після кожного запуску тестів. Правильно — `**/__pycache__`.
</details>

### Приклад 2.4. Порядок рядків і виняток `!`

Той самий набір рядків у різному порядку дає різний результат: перемагає **останній** рядок, що збігся.

In [ ]:
GOOD_ORDER = [".env.*", "!.env.example"]
BAD_ORDER = ["!.env.example", ".env.*"]

In [ ]:
for title, rules in (("спершу .env.*, потім виняток", GOOD_ORDER), ("спершу виняток, потім .env.*", BAD_ORDER)):
    print(f"{title:<30} .env.example виключено: {excluded('.env.example', rules)}")
assert excluded(".env.example", GOOD_ORDER) is False and excluded(".env.example", BAD_ORDER) is True

Виняток має стояти **після** загального правила — інакше наступний рядок знову виключає файл. Так само в `.gitignore`.

### Приклад 2.5. Контекст збірки `news_hub`

Пройдемо всі файли проєкту (`rglob("*")`) і залишимо ті, що **не** виключені, — це те, що `docker build` передасть Docker і що скопіює `COPY . .`.

In [ ]:
def build_context(root: Path, patterns: list[str]) -> list[str]:
    """Шляхи (через /) файлів, які потраплять у контекст збірки."""
    return sorted(p.relative_to(root).as_posix() for p in root.rglob("*")
                  if p.is_file() and not excluded(p.relative_to(root).as_posix(), patterns))

In [ ]:
from collections import Counter

context = build_context(PROJECT, DOCKERIGNORE)
size = sum((PROJECT / p).stat().st_size for p in context)
print(f"{len(context)} файлів, {size / 1024:.0f} кБ")
print(Counter(p.split("/")[0] if "/" in p else "(корінь)" for p in context))
for path in context[:8]:
    print(" ", path)
assert "news_hub/api.py" in context and "Dockerfile" not in context
assert not any(p.startswith(("tests/", ".venv/")) or p == ".env" or "__pycache__" in p for p in context)

35 файлів і пара сотень кілобайт: пакет `news_hub/` (26 файлів), `migrations/`, знімок новин у `data/` і два файли в корені (`alembic.ini`, `requirements.txt`). Справжній Docker на цій папці (`COPY . /ctx` + `find -type f`) дає той самий список — перевіримо в прикладі 8.2.

### Приклад 2.6. Папка студента з `.venv` і `.env`

У студента після уроку 48 поруч з кодом лежать `.venv` (сотні МБ) і `.env` з ключами. Змоделюємо таку папку в `docker_lab/` і порівняємо контекст **без** `.dockerignore` і **з** ним.

In [ ]:
def make_student_folder(root: Path) -> None:
    """Маленька копія ситуації: код, .env з ключем і «важкий» .venv (8 МБ замість сотень)."""
    (root / "news_hub").mkdir(parents=True, exist_ok=True)
    (root / "news_hub" / "api.py").write_text("app = 'FastAPI'\n")
    (root / ".env").write_text("GEMINI_API_KEY=AIza-FAKE-KEY-FOR-LESSON\n")
    venv = root / ".venv" / "lib" / "python3.12" / "site-packages"
    venv.mkdir(parents=True, exist_ok=True)
    for number in range(8):
        (venv / f"package_{number}.so").write_bytes(b"\0" * 1024 * 1024)


def context_size(root: Path, patterns: list[str]) -> tuple[int, int]:
    files = build_context(root, patterns)
    return len(files), sum((root / f).stat().st_size for f in files)

In [ ]:
student = LAB / "student_folder"
make_student_folder(student)
for title, rules in (("без .dockerignore", []), ("з .dockerignore news_hub", DOCKERIGNORE)):
    count, total = context_size(student, rules)
    print(f"{title:<26} {count:>2} файлів, {total / 1024:>7.1f} кБ")
assert context_size(student, DOCKERIGNORE)[0] == 1

Без `.dockerignore` у контекст іде все: 8 МБ «бібліотек» і `.env` з ключем. З ним — лише `news_hub/api.py`. На справжній папці студента різниця була 1.17 ГБ проти 328 кБ і 27 секунд лише на передачу контексту (книга, «Рефакторинг 1»).

### Задача 1. Яке правило виключило файл?

Docker не каже, **чому** файл не потрапив в образ. Напиши `why_excluded(path, patterns)` → текст **останнього** рядка `.dockerignore`, що збігся з `path` (разом з `!`, якщо це виняток), або `None`, якщо жоден не збігся. Порожні рядки й коментарі пропускай; `pattern_to_regex` уже є.

Приклад: для `.env.example` відповідь — `"!.env.example"`, для `news_hub/api.py` — `None`.

In [ ]:
def why_excluded(path: str, patterns: list[str]) -> str | None:
    """Останній рядок .dockerignore, що збігся з path (з «!», якщо виняток), або None."""
    # BEGIN SOLUTION
    last = None
    for raw in patterns:
        line = raw.strip()
        if not line or line.startswith("#"):
            continue
        if pattern_to_regex(line.removeprefix("!")).match(path):
            last = line
    return last
    # END SOLUTION

In [ ]:
answers = {path: why_excluded(path, DOCKERIGNORE) for path in
           [".env", ".env.example", "news_hub/api.py", "tests/unit/test_models.py", "news_hub/bot/__pycache__/x.pyc"]}
for path, rule in answers.items():
    print(f"{path:<32} ← {rule}")
assert answers[".env"] == ".env", f"для .env очікується рядок '.env', а отримано {answers['.env']!r}"
assert answers[".env.example"] == "!.env.example", "останнім збігся виняток — поверни його разом з «!»"
assert answers["news_hub/api.py"] is None, "жоден рядок не збігся — має бути None"
assert answers["tests/unit/test_models.py"] == "tests/"
assert answers["news_hub/bot/__pycache__/x.pyc"] == "**/*.py[cod]", "збіглися два рядки — потрібен останній"
for path in CASES:                                             # узгоджено з excluded
    rule = why_excluded(path, DOCKERIGNORE)
    assert (rule is not None and not rule.startswith("!")) == excluded(path, DOCKERIGNORE), path
print("✅ Задача 1 розв'язана")

---
## 3. Кеш шарів

Для кожної інструкції Docker шукає в кеші шар, зібраний з **тих самих вхідних даних** поверх **того самого** попереднього шару. Як тільки одна інструкція виконується заново, кеш для **всіх наступних** уже не діє.

```text
змінено news_hub/models.py:

  FROM python:3.12-slim        CACHED   образ той самий
  COPY requirements.txt .      CACHED   файл не змінився
  RUN pip install …            CACHED   попередній шар з кешу, інструкція та сама
  COPY . .                     RUN      models.py змінився        ← кеш обірвано тут
  USER, ENV, CMD …             RUN      попередній шар новий (метадані — миттєво)
```

### Приклад 3.1. `cache_report`: що виконається заново

Модель: вхідні дані `COPY <файли> <куди>` — ці файли (`.` — будь-який файл контексту); для решти інструкцій — лише сам текст (він не змінився). Щойно один `COPY` «зламався», усе нижче — `RUN`.

In [ ]:
def cache_report(steps: list[tuple[str, str]], changed: set[str]) -> list[str]:
    """CACHED / RUN для кожної інструкції, якщо змінились файли changed."""
    report, broken = [], False
    for name, args in steps:
        if name == "COPY" and not broken:
            sources = args.split()[:-1]
            broken = "." in sources or bool(changed & set(sources))
        report.append("RUN" if broken else "CACHED")
    return report


def show(title: str, steps: list[tuple[str, str]], changed: set[str]) -> list[str]:
    report = cache_report(steps, changed)
    print(title)
    for (name, args), status in zip(steps, report):
        print(f"  {status:<6} {name} {args[:50]}")
    return report

In [ ]:
good = show("правильний порядок, змінено models.py:", steps, {"news_hub/models.py"})
pip_index = [n for n, _ in steps].index("RUN")
assert good[pip_index] == "CACHED" and good[-1] == "RUN"

`pip install` — з кешу: залежності не перевстановлюються. Заново — лише `COPY . .` і метадані після нього. Виміряно на `news_hub`: **0.6 с** на перезбирання.

### Приклад 3.2. `COPY . .` перед `pip install`

🔮 **Прогноз:** той самий змінений `models.py`, але `COPY . .` стоїть **перед** `pip install`. Що буде з `pip install`?

In [ ]:
BAD = parse_dockerfile("""
FROM python:3.12-slim
WORKDIR /app
COPY . .
RUN pip install --no-cache-dir -r requirements.txt
CMD ["uvicorn", "news_hub.api:app", "--host", "0.0.0.0", "--port", "8000"]
""")

In [ ]:
bad = show("COPY . . перед pip install, змінено models.py:", BAD, {"news_hub/models.py"})
assert bad[3] == "RUN"

<details>
<summary>Відповідь</summary>

`pip install` виконується заново: попередній шар (`COPY . .`) новий, тож кеш уже не діє. На `news_hub` — **39.9 с** на кожну зміну коду замість 0.6 с.
</details>

### Приклад 3.3. Змінився `requirements.txt`

In [ ]:
req = show("правильний порядок, змінено requirements.txt:", steps, {"requirements.txt"})
assert req[pip_index] == "RUN" and req[1] == "CACHED"

Тепер заново все від `COPY requirements.txt .` — і це правильно: нові залежності треба поставити. Правило порядку: **від того, що змінюється рідко, до того, що змінюється щодня**.

### Приклад 3.4. Ключі кешу з вмісту файлів

Docker порівнює не «чи змінився файл» за назвою, а **хеш вмісту**. Ключ шару = хеш(ключ попереднього шару + інструкція + вміст файлів для `COPY`). Порахуємо такі ключі для справжніх файлів `news_hub`, а потім «змінимо» `models.py` — не на диску, а в словнику `overrides`.

In [ ]:
import hashlib


def layer_keys(steps: list[tuple[str, str]], files: list[str], root: Path,
               overrides: dict[str, bytes] | None = None) -> list[str]:
    """Ключ кешу кожної інструкції: хеш попереднього ключа, тексту інструкції і вмісту файлів COPY."""
    overrides = overrides or {}
    keys, previous = [], ""
    for name, args in steps:
        digest = hashlib.sha256(f"{previous}|{name} {args}".encode())
        if name == "COPY":
            sources = args.split()[:-1]
            for path in files:
                if "." in sources or path in sources:
                    digest.update(path.encode() + overrides.get(path, (root / path).read_bytes()))
        previous = digest.hexdigest()[:12]
        keys.append(previous)
    return keys

In [ ]:
before = layer_keys(steps, context, PROJECT)
after = layer_keys(steps, context, PROJECT,
                   overrides={"news_hub/models.py": (PROJECT / "news_hub/models.py").read_bytes() + b"# x\n"})
for (name, args), old, new in zip(steps, before, after):
    print(f"{name:<11} {old}  {new}  {'CACHED' if old == new else 'RUN'}")
assert before[pip_index] == after[pip_index] and before[-1] != after[-1]

Той самий результат, що в прикладі 3.1, але тепер із «справжніми» ключами: до `COPY . .` ключі однакові, після — усі різні, бо кожен ключ містить попередній. Звідси і правило про `.dockerignore`: якби в контекст потрапляв `__pycache__`, його вміст змінювався б після кожного запуску тестів — і `COPY . .` щоразу виконувався б заново.

---
## 4. Секрети й змінні середовища

### Приклад 4.1. Секрет у `ENV`

`ENV` у Dockerfile записується в метадані образу — `docker history` і `docker inspect` покажуть значення кожному. Знайдемо змінні з `ENV` чи `ARG`, у назві яких є `KEY`, `SECRET`, `TOKEN` або `PASSWORD` і яким задано значення.

In [ ]:
SECRET_WORDS = ("KEY", "SECRET", "TOKEN", "PASSWORD")


def secrets_in_dockerfile(steps: list[tuple[str, str]]) -> list[str]:
    """Імена змінних ENV/ARG зі значенням, схожих на секрет."""
    found = []
    for name, args in steps:
        if name not in ("ENV", "ARG"):
            continue
        for key, value in parse_env(args).items():
            if value and any(word in key.upper() for word in SECRET_WORDS):
                found.append(key)
    return found


LEAKY = parse_dockerfile("ENV PYTHONUNBUFFERED=1 GEMINI_API_KEY=AIza-demo\nENV BOT_TOKEN 123:ABC\nARG PIP_INDEX_URL\nENV DEBUG=0")

In [ ]:
print("news_hub:", secrets_in_dockerfile(steps))
print("leaky:   ", secrets_in_dockerfile(LEAKY))
assert secrets_in_dockerfile(steps) == []
assert secrets_in_dockerfile(LEAKY) == ["GEMINI_API_KEY", "BOT_TOKEN"]

У `news_hub` секретів у Dockerfile немає. У «дірявому» — два: обидві форми `ENV` спіймано, `ARG PIP_INDEX_URL` без значення пропущено.

### Приклад 4.2. Що бачить кожен, хто має образ

Метадані образу — JSON (його показує `docker inspect`). Зберемо такий самий фрагмент з інструкцій: `Env`, `User`, `Cmd`, `ExposedPorts`. Усе тут — **публічне** для власника образу.

In [ ]:
def image_config(steps: list[tuple[str, str]]) -> dict[str, object]:
    """Фрагмент Config з docker inspect: те, що інструкції записують у метадані."""
    config: dict[str, object] = {"Env": [], "User": "", "Cmd": None, "ExposedPorts": {}, "WorkingDir": ""}
    for key, value in image_env(steps).items():
        config["Env"].append(f"{key}={value}")
    for name, args in steps:
        if name == "USER":
            config["User"] = args
        elif name == "CMD":
            config["Cmd"] = command_of(args)
        elif name == "EXPOSE":
            config["ExposedPorts"][f"{args}/tcp"] = {}
        elif name == "WORKDIR":
            config["WorkingDir"] = args
    return config

In [ ]:
print(json.dumps(image_config(steps), indent=2, ensure_ascii=False))
print("\nдірявий образ:", image_config(LEAKY)["Env"])

У дірявому образі `GEMINI_API_KEY=AIza-demo` лежить у метаданих відкритим текстом. Реєстр, колега, CI — усі, хто отримав образ, отримали й ключ. Справжній `docker inspect` — у прикладі 8.4.

### Приклад 4.3. `RUN rm .env` не допомагає: шари як архіви

Кожен шар — окремий tar-архів. Видалення файлу в наступному шарі — це лише **позначка** `.wh.<ім'я>` («whiteout»): у зібраній файловій системі файлу не видно, але попередній архів лишився в образі. Змоделюємо два шари справжніми tar-файлами.

In [ ]:
import io
import tarfile


def write_layer(path: Path, files: dict[str, bytes]) -> None:
    with tarfile.open(path, "w") as tar:
        for name, data in files.items():
            info = tarfile.TarInfo(name)
            info.size = len(data)
            tar.addfile(info, io.BytesIO(data))


def merged_view(layers: list[Path]) -> dict[str, bytes]:
    """Що бачить контейнер: шари згори вниз, .wh.<ім'я> видаляє файл з нижчих шарів."""
    files: dict[str, bytes] = {}
    for layer in layers:
        with tarfile.open(layer) as tar:
            for member in tar.getmembers():
                folder, _, base = member.name.rpartition("/")
                if base.startswith(".wh."):
                    files.pop(f"{folder}/{base[4:]}".lstrip("/"), None)
                else:
                    files[member.name] = tar.extractfile(member).read()
    return files

In [ ]:
layer1, layer2 = LAB / "layer1_copy.tar", LAB / "layer2_rm_env.tar"
write_layer(layer1, {"app/api.py": b"app = 'FastAPI'\n", "app/.env": b"GEMINI_API_KEY=AIza-FAKE-KEY-FOR-LESSON\n"})
write_layer(layer2, {"app/.wh..env": b""})                      # так виглядає RUN rm /app/.env

print("контейнер бачить:", sorted(merged_view([layer1, layer2])))
with tarfile.open(layer1) as tar:
    print("а в шарі 1 лежить:", tar.extractfile("app/.env").read().decode().strip())
assert "app/.env" not in merged_view([layer1, layer2])

`ls /app` у контейнері `.env` не покаже, але архів першого шару в образі — і будь-хто дістане ключ через `docker save`. Висновок книги: секрет **не має потрапляти в жоден шар**. Справжній `docker save` з таким образом — приклад 8.4.

### Приклад 4.4. `--env-file` бере значення буквально

`python -m news_hub.security` друкує рядки для `export` у shell — з одинарними лапками, бо в bcrypt-хеші є `$`. Compose знімає лапки, а `docker run --env-file` — ні. Напишемо обидва читачі `.env`.

In [ ]:
ENV_TEXT = """
# рядки від python -m news_hub.security
ADMIN_PASSWORD_HASH='$2b$12$eBDJbnUjhmlfW0bqlm9n1uCzKlO6dQoHzU6o1v2Ej2m4kB7l1Qv6y'
JWT_SECRET='4GnN8DU8jRhAk6z09C-avMa3mSJd'
"""


def read_docker(text: str) -> dict[str, str]:
    """Як docker run --env-file: KEY=значення як є, разом з лапками."""
    return dict(line.split("=", 1) for line in text.splitlines() if line.strip() and not line.startswith("#"))


def read_compose(text: str) -> dict[str, str]:
    """Як env_file у Compose: однакові лапки навколо значення знімаються."""
    values = {}
    for key, value in read_docker(text).items():
        if len(value) >= 2 and value[0] == value[-1] and value[0] in "'\"":
            value = value[1:-1]
        values[key] = value
    return values


def app_starts(env: dict[str, str]) -> bool:
    return env["ADMIN_PASSWORD_HASH"].startswith("$2")        # спрощене правило load_admin_settings (урок 47)

🔮 **Прогноз:** з яким читачем застосунок стартує?

In [ ]:
print("docker run --env-file:", app_starts(read_docker(ENV_TEXT)), read_docker(ENV_TEXT)["ADMIN_PASSWORD_HASH"][:12])
print("compose env_file:     ", app_starts(read_compose(ENV_TEXT)), read_compose(ENV_TEXT)["ADMIN_PASSWORD_HASH"][:12])
assert not app_starts(read_docker(ENV_TEXT)) and app_starts(read_compose(ENV_TEXT))

<details>
<summary>Відповідь</summary>

Лише з Compose. `docker run --env-file` передав застосунку значення **разом з лапкою** `'$2b…`, і перевірка хешу не пройшла. Справжній запуск: `RuntimeError: ADMIN_PASSWORD_HASH — bcrypt-хеш ($2b$…), а не пароль`, контейнер `Exited (3)`. Добре, що застосунок впав одразу (fail fast), а не працював без адмінки. Для `--env-file` є окремий режим: `python -m news_hub.security --env-file` (без лапок).
</details>

### Приклад 4.5. Що робить shell з тим самим рядком

Одинарні лапки в bash — «усе буквально», тому `source` того самого файлу дає правильний хеш. Перевіримо справжнім bash: запишемо файл і прочитаємо змінну з дочірнього Python.

In [ ]:
ENV_PATH = LAB / "admin.env"
ENV_PATH.write_text(ENV_TEXT)
SOURCE_AND_PRINT = f"set -a; source {ENV_PATH}; set +a; " \
                   "python3 -c \"import os; print(os.environ['ADMIN_PASSWORD_HASH'])\""

In [ ]:
from_bash = subprocess.run(["bash", "-c", SOURCE_AND_PRINT], capture_output=True, text=True).stdout.strip()
print("bash source:", from_bash[:20], "…")
assert from_bash == read_compose(ENV_TEXT)["ADMIN_PASSWORD_HASH"]

bash і Compose читають файл однаково, `docker run --env-file` — інакше. Тому для кожного «споживача» потрібен свій формат рядка.

### Задача 2. Рядок `.env` для потрібного споживача

Напиши `env_file_line(key, value, target)`:

- `target="docker"` — для `docker run --env-file`: `KEY=значення` **без лапок** (Docker бере все буквально);
- `target="shell"` — для `source` у bash: значення в **одинарних лапках**; одинарну лапку всередині значення записуй як `'\''` (закрити лапки, екранована лапка, відкрити знову).

Якщо в значенні є перенос рядка — `ValueError` (жоден з форматів його не підтримує).

In [ ]:
def env_file_line(key: str, value: str, target: str) -> str:
    """Рядок .env для docker run --env-file ("docker") або для source у bash ("shell")."""
    # BEGIN SOLUTION
    if "\n" in value:
        raise ValueError("перенос рядка в значенні не підтримується")
    if target == "docker":
        return f"{key}={value}"
    if target == "shell":
        return f"{key}='" + value.replace("'", "'\\''") + "'"
    raise ValueError(f"невідомий target: {target}")
    # END SOLUTION

In [ ]:
TRICKY_VALUES = {"ADMIN_PASSWORD_HASH": "$2b$12$eBDJbnUjhmlfW0bqlm9n1u", "GREETING": "it's $HOME & more"}
docker_lines = [env_file_line(k, v, "docker") for k, v in TRICKY_VALUES.items()]
shell_lines = [env_file_line(k, v, "shell") for k, v in TRICKY_VALUES.items()]
assert all(isinstance(line, str) for line in docker_lines + shell_lines), "функція має повертати рядок"
print("\n".join(docker_lines), "\n---\n" + "\n".join(shell_lines))
assert read_docker("\n".join(docker_lines)) == TRICKY_VALUES, "для docker значення має бути без лапок, як є"

(LAB / "task2.env").write_text("\n".join(shell_lines) + "\n")
script = (f"set -a; source {LAB / 'task2.env'}; set +a; "
          "python3 -c \"import os, json; print(json.dumps({k: os.environ[k] for k in ('ADMIN_PASSWORD_HASH', 'GREETING')}))\"")
loaded = json.loads(subprocess.run(["bash", "-c", script], capture_output=True, text=True).stdout or "{}")
assert loaded == TRICKY_VALUES, f"bash прочитав інакше: {loaded}"
try:
    env_file_line("A", "рядок 1\nрядок 2", "docker")
    raise AssertionError("перенос рядка має давати ValueError")
except ValueError:
    pass
print("✅ Задача 2 розв'язана")

---
## 5. Сигнали: чому `docker stop` чекає 10 секунд

`docker stop` надсилає `SIGTERM` процесу з **PID 1** і чекає 10 с, а потім — `SIGKILL`. Shell-форма `CMD uvicorn …` робить PID 1 оболонку `/bin/sh -c`, а exec-форма `CMD ["uvicorn", …]` — сам застосунок. Відтворимо це без Docker: «сервер» ловить `SIGTERM` і пише в журнал, що закриває з'єднання.

### Приклад 5.1. «Сервер» і зупинка «як Docker»

Перша клітинка записує `server.py` і функцію `stop_like_docker`, яка запускає команду, надсилає `SIGTERM` **першому** процесу (як `docker stop` — PID 1) і дивиться, що сталося з сервером.

In [ ]:
import signal
import sys
import textwrap
import time

SERVER = LAB / "server.py"
SERVER.write_text(textwrap.dedent("""
    import os, signal, sys, time
    from pathlib import Path
    here = Path(__file__).parent
    (here / "server.pid").write_text(str(os.getpid()))
    def stop(signum, frame):
        with open(here / "server.log", "a") as log:
            log.write("SIGTERM: закриваю з'єднання\\n")
        sys.exit(0)
    signal.signal(signal.SIGTERM, stop)
    while True:
        time.sleep(0.1)
"""))


def alive(pid: int) -> bool:
    try:
        os.kill(pid, 0)
        return True
    except ProcessLookupError:
        return False


def stop_like_docker(command: list[str]) -> dict[str, object]:
    """Запустити command, надіслати SIGTERM першому процесу (як docker stop — PID 1), подивитись, що сталось."""
    for name in ("server.pid", "server.log"):
        (LAB / name).unlink(missing_ok=True)
    top = subprocess.Popen(command)
    while not (LAB / "server.pid").exists():
        time.sleep(0.05)
    server = int((LAB / "server.pid").read_text())
    top.send_signal(signal.SIGTERM)
    top.wait(timeout=5)
    time.sleep(0.5)
    survived = alive(server)
    if survived:
        os.kill(server, signal.SIGKILL)            # як docker через 10 с
    log = (LAB / "server.log").read_text().strip() if (LAB / "server.log").exists() else ""
    return {"сервер = перший процес": top.pid == server, "журнал": log, "сервер пережив SIGTERM": survived}

### Приклад 5.2. Напряму — як exec-форма `CMD`

In [ ]:
direct = stop_like_docker([sys.executable, str(SERVER)])
print("напряму (exec-форма CMD)  ", direct)
assert direct["сервер = перший процес"] and direct["журнал"]

Сервер — перший процес, сам отримав `SIGTERM`, записав журнал і завершився.

### Приклад 5.3. Через оболонку — як shell-форма `CMD`

🔮 **Прогноз:** `["sh", "-c", "python server.py"]`. Чи буде рядок у журналі?

In [ ]:
shell = stop_like_docker(["sh", "-c", f"{sys.executable} {SERVER}"])
print("sh -c (shell-форма CMD)   ", shell)
assert not shell["журнал"] and shell["сервер пережив SIGTERM"]

<details>
<summary>Відповідь</summary>

Ні. Сигнал отримала оболонка і завершилась, а сервер навіть не дізнався про зупинку — він «пережив» `SIGTERM`, і ми вбили його `SIGKILL`, як Docker через 10 с (код 137). У контейнері це означає: lifespan FastAPI не виконався, з'єднання з Redis і базою обірвані.
</details>

### Приклад 5.4. Оболонка з `exec`

`exec` **замінює** оболонку програмою: той самий PID, але тепер це Python. Саме тому exec-форма `CMD` (або `exec uvicorn …` у скрипті запуску) — правильний спосіб.

In [ ]:
with_exec = stop_like_docker(["sh", "-c", f"exec {sys.executable} {SERVER}"])
print("sh -c exec …              ", with_exec)
assert with_exec["сервер = перший процес"] and with_exec["журнал"]

Як і напряму: сервер — перший процес і зупинився коректно. Справжній `docker stop` для `news_hub`: exec-форма — 1.8 с, код 0, «Application shutdown complete»; shell-форма — 10.2 с, код 137. На справжньому Docker повторимо в прикладі 8.5.

### Приклад 5.5. Код виходу 128 + N

Процес, убитий сигналом N, у shell має код `128 + N`: `143` після `SIGTERM` (15), `137` після `SIGKILL` (9). Python показує те саме як від'ємне число: `-15`, `-9`.

In [ ]:
EXIT_CODES = """
sleep 30 & kill -TERM $!; wait $!; echo "SIGTERM → $?"
sleep 30 & kill -KILL $!; wait $!; echo "SIGKILL → $?"
sleep 30 & kill -INT  $!; wait $!; echo "SIGINT  → $?"
"""

In [ ]:
print(subprocess.run(["bash", "-c", EXIT_CODES], capture_output=True, text=True).stdout)
victim = subprocess.Popen(["sleep", "30"])
victim.kill()
print("Python бачить SIGKILL як", victim.wait())

Тепер `Exited (137)` у `docker ps -a` читається одразу: «убито `SIGKILL`» — найчастіше тому, що на `SIGTERM` процес за 10 с не завершився.

### Приклад 5.6. Пауза `docker stop -t`

`docker stop -t N`: `SIGTERM`, чекати N секунд, потім `SIGKILL`. Напишемо те саме для звичайного процесу й порівняємо процес, що реагує на `SIGTERM`, з процесом, який його ігнорує (як `sh` у ролі PID 1).

In [ ]:
def stop_with_grace(command: list[str], grace: float) -> tuple[float, int]:
    """SIGTERM → чекати grace секунд → SIGKILL. Повертає (скільки тривала зупинка, код виходу)."""
    process = subprocess.Popen(command)
    time.sleep(1.0)                                              # дати процесу стартувати й поставити обробник
    started = time.perf_counter()
    process.terminate()
    try:
        code = process.wait(timeout=grace)
    except subprocess.TimeoutExpired:
        process.kill()
        code = process.wait()
    return round(time.perf_counter() - started, 1), code


GRACEFUL = [sys.executable, "-c", "import signal, sys, time; signal.signal(signal.SIGTERM, lambda *a: sys.exit(0)); time.sleep(60)"]
IGNORES = [sys.executable, "-c", "import signal, time; signal.signal(signal.SIGTERM, signal.SIG_IGN); time.sleep(60)"]

In [ ]:
print("реагує на SIGTERM:  (секунд, код) =", stop_with_grace(GRACEFUL, grace=2))
print("ігнорує SIGTERM:    (секунд, код) =", stop_with_grace(IGNORES, grace=2))

Перший завершився майже миттєво з кодом 0. Другий чекав усю паузу (2 с) і отримав `SIGKILL` (`-9`, у Docker — 137). У Docker пауза — 10 с, тому «повільний `docker stop`» — майже завжди процес, до якого не дійшов `SIGTERM`.

---
## 6. Здоров'я і порти

| Ендпоінт | Питання | Хто питає | Не так — що робити |
|---|---|---|---|
| `GET /health` | процес відповідає? | `HEALTHCHECK` образу | перезапустити контейнер |
| `GET /health/ready` | база й Redis доступні? | балансувальник, Compose (урок 50) | не слати запитів, почекати |

### Приклад 6.1. Відповідь `/health/ready`

Логіка ендпоінта `ready` з `news_hub/api.py` без FastAPI: назовні — лише `ok` або **назва** винятку (адреса бази й текст помилки лишаються в журналі), код 200 лише коли все `ok`.

In [ ]:
def readiness(checks: dict[str, str]) -> tuple[int, dict[str, object]]:
    ok = all(value == "ok" for value in checks.values())
    return (200 if ok else 503), {"status": "ok" if ok else "unavailable", "checks": checks}

In [ ]:
for checks in ({"database": "ok", "redis": "ok"}, {"database": "ok", "redis": "ConnectionError"}):
    print(*readiness(checks))

Саме такі відповіді в книзі: контейнер API без Redis у своїй мережі — `503` з `"redis": "ConnectionError"` (`localhost` у контейнері — це сам контейнер), у спільній мережі — `200`.

### Приклад 6.2. Як `HEALTHCHECK` вирішує «healthy» чи «unhealthy»

Docker виконує перевірку кожні `--interval` секунд. Невдачі протягом `--start-period` не рахуються; `--retries` невдач **поспіль** після нього — `unhealthy`; будь-який успіх — `healthy`. Змоделюємо для параметрів проєкту (`30s`, `start-period=30s`, `retries=3`).

In [ ]:
def health_states(probes: list[bool], interval: int = 30, start_period: int = 30, retries: int = 3) -> list[str]:
    """Статус контейнера після кожної перевірки (True — /health відповів)."""
    status, failures, states = "starting", 0, []
    for number, ok in enumerate(probes, 1):
        in_start_period = number * interval <= start_period
        if ok:
            status, failures = "healthy", 0
        elif not in_start_period:
            failures += 1
            if failures >= retries:
                status = "unhealthy"
        states.append(status)
    return states


SLOW_START = [False, True, True]                         # імпорт бібліотек — кілька секунд
FROZEN = [True, True, False, False, False]               # процес «завис»

In [ ]:
print("повільний старт:", health_states(SLOW_START))
print("завис:          ", health_states(FROZEN))
print("без start-period:", health_states(SLOW_START, start_period=0, retries=1))
assert health_states(FROZEN)[-1] == "unhealthy" and health_states(SLOW_START)[-1] == "healthy"

Повільний старт не робить контейнер `unhealthy` — для цього й `--start-period`. Без нього й з `retries=1` той самий старт уже на першій перевірці дав би `unhealthy`: оркестратор перезапустив би контейнер, який просто ще не встиг стартувати. Чому `HEALTHCHECK` питає `/health`, а не `/health/ready`: якщо впаде база, перезапуск API її не полагодить.

### Приклад 6.3. `-p`: який порт відкрито й для кого

`-p 8000:8000` відкриває порт контейнера на **всіх** адресах комп'ютера, `-p 127.0.0.1:8000:8000` — лише для самого комп'ютера. Розберемо формат `[адреса:]порт_комп'ютера:порт_контейнера`.

In [ ]:
def parse_publish(spec: str) -> dict[str, object]:
    parts = spec.split(":")
    host_ip = parts[0] if len(parts) == 3 else "0.0.0.0"
    host_port, container_port = parts[-2], parts[-1]
    return {"адреса": host_ip, "порт комп'ютера": int(host_port), "порт контейнера": int(container_port),
            "видно з мережі": host_ip == "0.0.0.0"}

In [ ]:
for spec in ("8000:8000", "127.0.0.1:8000:8000", "127.0.0.1:18000:8000"):
    print(f"-p {spec:<22}", parse_publish(spec))

Усередині контейнера uvicorn слухає `0.0.0.0:8000` — інакше `-p` нікуди було б перенаправляти. А **хто** бачить порт ззовні, вирішує адреса в `-p`. Для розробки — `127.0.0.1:…` (урок 50, `docker-compose.dev.yml`).

---
## 7. Знайди помилку

Dockerfile нижче збирається, і `curl /health` відповідає `{"status":"ok"}`. Але в ньому три проблеми.

In [ ]:
FIND_BUG = """
FROM python:3.12-slim
WORKDIR /app
COPY . .
RUN pip install --no-cache-dir -r requirements.txt
RUN useradd --uid 10001 app
USER app
ENV GEMINI_API_KEY=AIzaSyD-course-demo-key-not-real-0000000
CMD uvicorn news_hub.api:app --host 0.0.0.0 --port 8000
"""

AS_ROOT = """
FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
CMD ["uvicorn", "news_hub.api:app", "--host", "0.0.0.0", "--port", "8000"]
"""

### Задача 3. Лінтер `Dockerfile`

Напиши `lint(text)` → множину кодів проблем:

- `"order"` — після зміни `news_hub/models.py` інструкція `RUN pip install …` виконається заново (використай `cache_report`);
- `"secret"` — секрет у `ENV`/`ARG` (`secrets_in_dockerfile`);
- `"shell-cmd"` — `CMD` не в exec-формі (не починається з `[`);
- `"root"` — немає інструкції `USER` (або остання `USER` — `root` чи `0`): застосунок працюватиме від root.

На `Dockerfile` проєкту `lint` має повернути порожню множину, на `FIND_BUG` — три коди, на `AS_ROOT` — лише `"root"`.

In [ ]:
def lint(text: str) -> set[str]:
    """Коди проблем Dockerfile: order, secret, shell-cmd, root."""
    # BEGIN SOLUTION
    parsed = parse_dockerfile(text)
    problems = set()
    report = cache_report(parsed, {"news_hub/models.py"})
    if any(name == "RUN" and "pip install" in args and status == "RUN"
           for (name, args), status in zip(parsed, report)):
        problems.add("order")
    if secrets_in_dockerfile(parsed):
        problems.add("secret")
    if any(name == "CMD" and not args.startswith("[") for name, args in parsed):
        problems.add("shell-cmd")
    users = [args.split(":")[0] for name, args in parsed if name == "USER"]
    if not users or users[-1] in ("root", "0"):
        problems.add("root")
    return problems
    # END SOLUTION

In [ ]:
results = {"find-bug": lint(FIND_BUG), "as-root": lint(AS_ROOT), "news_hub": lint(DOCKERFILE)}
for title, found in results.items():
    print(f"{title:<9}", sorted(found) if found is not None else found)
assert results["news_hub"] == set(), f"Dockerfile проєкту чистий, а знайдено {results['news_hub']}"
assert results["find-bug"] == {"order", "secret", "shell-cmd"}, f"FIND_BUG: {results['find-bug']}"
assert results["as-root"] == {"root"}, f"AS_ROOT: {results['as-root']}"
assert lint(DOCKERFILE.replace("USER app", "USER root")) == {"root"}, "USER root — теж root"
print("✅ Задача 3 розв'язана")

<details>
<summary>Відповідь і справжній вивід</summary>

```text
$ echo "# x" >> news_hub/models.py && time docker build -q -t findbug .
real    0m43.050s                       ← order: pip install на кожну зміну коду

$ docker history findbug --format '{{.CreatedBy}}' | grep GEMINI
ENV GEMINI_API_KEY=AIzaSyD-course-demo-key-n…    ← secret: ключ бачить кожен, хто має образ

$ time docker stop fb
real    0m10.262s
$ docker inspect -f '{{.State.ExitCode}}' fb
137                                     ← shell-cmd: SIGTERM не дійшов до uvicorn
```

`curl /health` жодної з цих проблем не показує: вони не в тому, **чи** працює застосунок, а в тому, як його збирають, що в ньому лежить і як він зупиняється. У проєкті це перевіряють тести `tests/docker/test_image.py` (`pytest -m docker`).
</details>

---
## 8. Справжній Docker

Якщо Docker встановлений (Docker Desktop чи Docker Engine), клітинки нижче повторюють розділи 2–5 на справжніх образах і контейнерах. Без Docker (Colab) кожна клітинка друкує, що пропущено, — а результат справжнього запуску наведено в кінці розділу.

Усе, що створюємо, має назви з префіксом `course-m5-l49-` і мітку `course-m5-l49`, і видаляється в прикладі 8.8. Образи будуємо з `python:3.12-slim` — Docker завантажить його один раз (≈ 50 МБ).

### Приклад 8.1. Помічники

In [ ]:
import socket
import urllib.request

PREFIX = "course-m5-l49-"
LABEL = "course-m5-l49=1"


def docker(*args: str, check: bool = True, timeout: int = 600) -> subprocess.CompletedProcess[str]:
    """Виконати docker …; stdout і stderr — у результаті."""
    result = subprocess.run(["docker", *args], capture_output=True, text=True, timeout=timeout)
    if check and result.returncode != 0:
        raise RuntimeError(f"docker {' '.join(args[:3])} …: {result.stderr.strip()[-500:]}")
    return result


def build(tag: str, context: Path, dockerfile: str, *extra: str) -> tuple[float, str]:
    """docker build з Dockerfile-текстом (кладемо його в docker_lab/); повертає (секунди, журнал збірки)."""
    path = LAB / f"{tag}.Dockerfile"
    path.write_text(dockerfile)
    started = time.perf_counter()
    result = docker("build", "--progress=plain", "--label", LABEL, "-t", PREFIX + tag, "-f", str(path), *extra, str(context))
    return round(time.perf_counter() - started, 1), result.stderr


def free_port() -> int:
    with socket.socket() as sock:
        sock.bind(("127.0.0.1", 0))
        return sock.getsockname()[1]


BASE_READY = False
if DOCKER:
    BASE_READY = docker("image", "inspect", "python:3.12-slim", check=False).returncode == 0 or \
        docker("pull", "-q", "python:3.12-slim", check=False).returncode == 0

In [ ]:
if DOCKER:
    print(docker("version", "--format", "клієнт {{.Client.Version}}, сервер {{.Server.Version}}").stdout.strip())
    print("python:3.12-slim:", "є" if BASE_READY else "не вдалося завантажити (немає мережі?)")
else:
    print("⏭ Docker недоступний — приклади 8.2–8.8 пропущено; вивід справжнього запуску — наприкінці розділу.")

### Приклад 8.2. Контекст збірки: Python проти Docker

Збираємо образ, який просто копіює **весь контекст** `news_hub` у `/ctx` (`.dockerignore` — той самий, з кореня контексту), і порівнюємо список файлів у контейнері зі списком `context` з прикладу 2.5.

In [ ]:
CONTEXT_PROBE = """
FROM python:3.12-slim
COPY . /ctx
CMD ["find", "/ctx", "-type", "f"]
"""

In [ ]:
if DOCKER and BASE_READY:
    seconds, _ = build("context", PROJECT, CONTEXT_PROBE)
    in_image = sorted(line.removeprefix("/ctx/") for line in docker("run", "--rm", PREFIX + "context").stdout.split())
    print(f"зібрано за {seconds} с; файлів в образі: {len(in_image)}, за моделлю Python: {len(context)}")
    print("однакові списки:", in_image == context)
    assert in_image == context, set(in_image) ^ set(context)
else:
    print("⏭ пропущено: немає Docker")

Списки збігаються файл у файл: наша модель `.dockerignore` (приклади 2.1–2.2) поводиться як Docker.

### Приклад 8.3. Кеш шарів наживо

Щоб не залежати від мережі, «важку» установку залежностей замінимо на `RUN sleep 3` (залежить лише від `requirements.txt`). Збираємо двічі з правильним порядком і двічі з неправильним, між збірками змінюючи лише `app.py`.

In [ ]:
CACHE_GOOD = """
FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN sleep 3 && echo "встановлено залежності з requirements.txt"
COPY . .
"""
CACHE_BAD = CACHE_GOOD.replace("COPY requirements.txt .\n", "COPY . .\n", 1).rsplit("COPY . .", 1)[0]


def cache_run(dockerfile: str, tag: str) -> list[float]:
    """Збірка, зміна app.py, ще збірка. Повертає [секунди другої збірки, чи «pip» узято з кешу]."""
    context = LAB / f"cache_{tag}"
    context.mkdir(exist_ok=True)
    (context / "requirements.txt").write_text("fastapi\n")
    (context / "app.py").write_text("print('v1')\n")
    build(tag, context, dockerfile)
    (context / "app.py").write_text(f"print('v2 {time.time()}')\n")
    seconds, log = build(tag, context, dockerfile)
    step_ids = {line.split()[0] for line in log.splitlines() if "RUN sleep 3" in line}   # «#7 [3/4] RUN sleep 3 …»
    install_cached = any(f"{step} CACHED" in log for step in step_ids)                    # «#7 CACHED»
    return [seconds, install_cached]

In [ ]:
if DOCKER and BASE_READY:
    print(CACHE_BAD)
    good_run, bad_run = cache_run(CACHE_GOOD, "cache-good"), cache_run(CACHE_BAD, "cache-bad")
    print(f"правильний порядок: друга збірка {good_run[0]} с, «pip» з кешу: {good_run[1]}")
    print(f"COPY . . спочатку:  друга збірка {bad_run[0]} с, «pip» з кешу: {bad_run[1]}")
    assert good_run[1] and not bad_run[1]
else:
    print("⏭ пропущено: немає Docker")

У правильному порядку крок установки — `CACHED`, друга збірка — близько секунди (накладні витрати самого Docker). У неправильному — `sleep 3` виконується щоразу, і збірка триває 4–5 с. На `news_hub` з реальним `pip install` різниця 0.6 с проти 39.9 с.

### Приклад 8.4. Секрет в `ENV` і секрет у видаленому файлі

Два «дірявих» образи: ключ в `ENV` і `.env`, скопійований і потім видалений `RUN rm`. Дістанемо обидва ключі так, як це зробив би будь-хто з образом: `docker inspect` і `docker save`.

In [ ]:
LEAKY_ENV = """
FROM python:3.12-slim
ENV GEMINI_API_KEY=AIza-FAKE-KEY-FOR-LESSON
"""
LEAKY_RM = """
FROM python:3.12-slim
COPY .env /app/.env
RUN rm /app/.env
"""


def find_in_saved_image(image: str, needle: bytes) -> list[str]:
    """docker save → tar з шарами; повертає імена файлів у шарах, що містять needle."""
    archive = LAB / "saved.tar"
    docker("save", "-o", str(archive), image)
    hits = []
    with tarfile.open(archive) as outer:
        for member in outer.getmembers():
            if not member.isfile():
                continue
            data = outer.extractfile(member).read()
            try:                                            # шар — tar (часто стиснутий gzip); JSON-файли пропускаємо
                with tarfile.open(fileobj=io.BytesIO(data), mode="r:*") as layer:
                    for item in layer.getmembers():
                        if item.isfile() and needle in layer.extractfile(item).read():
                            hits.append(item.name)
            except tarfile.TarError:
                continue
    archive.unlink()
    return hits

In [ ]:
if DOCKER and BASE_READY:
    build("leaky-env", LAB, LEAKY_ENV)
    print("docker inspect Env:", docker("inspect", "-f", "{{.Config.Env}}", PREFIX + "leaky-env").stdout.strip())
    (LAB / "rm_context").mkdir(exist_ok=True)
    (LAB / "rm_context" / ".env").write_text("GEMINI_API_KEY=AIza-FAKE-KEY-FOR-LESSON\n")
    build("leaky-rm", LAB / "rm_context", LEAKY_RM)
    print("ls /app у контейнері:", docker("run", "--rm", PREFIX + "leaky-rm", "ls", "-A", "/app").stdout.split())
    hits = find_in_saved_image(PREFIX + "leaky-rm", b"AIza-FAKE-KEY")
    print("ключ знайдено в шарі docker save:", hits)
    assert hits
else:
    print("⏭ пропущено: немає Docker")

`docker inspect` показав ключ у `Env`. У другому образі `ls /app` порожній — але в архіві шарів (`docker save`) файл `app/.env` з ключем на місці. Точно як у моделі з прикладу 4.3.

### Приклад 8.5. `docker stop`: exec-форма проти shell-форми

Один і той самий «сервер» (ловить `SIGTERM`, друкує рядок і завершується з кодом 0) з двома формами `CMD`. Щоб не чекати 10 с, зупиняємо з `-t 3`.

In [ ]:
STOP_APP = textwrap.dedent("""
    import signal, sys, time
    def stop(signum, frame):
        print("SIGTERM: закриваю з'єднання", flush=True)
        sys.exit(0)
    signal.signal(signal.SIGTERM, stop)
    print("сервер працює", flush=True)
    while True:
        time.sleep(0.1)
""")
EXEC_FORM = 'FROM python:3.12-slim\nCOPY stop_app.py /app/\nCMD ["python", "/app/stop_app.py"]\n'
SHELL_FORM = "FROM python:3.12-slim\nCOPY stop_app.py /app/\nCMD python /app/stop_app.py\n"


def docker_stop_timing(tag: str, dockerfile: str) -> dict[str, object]:
    context = LAB / "stop_context"
    context.mkdir(exist_ok=True)
    (context / "stop_app.py").write_text(STOP_APP)
    build(tag, context, dockerfile)
    name = PREFIX + tag
    docker("rm", "-f", name, check=False)
    docker("run", "-d", "--name", name, PREFIX + tag)
    time.sleep(1.5)
    pid1 = docker("exec", name, "cat", "/proc/1/cmdline").stdout.replace("\0", " ").strip()
    started = time.perf_counter()
    docker("stop", "-t", "3", name)
    seconds = round(time.perf_counter() - started, 1)
    code = docker("inspect", "-f", "{{.State.ExitCode}}", name).stdout.strip()
    logs = docker("logs", name).stdout.strip().splitlines()
    docker("rm", name)
    return {"PID 1": pid1, "секунд": seconds, "код": int(code), "журнал": logs}

In [ ]:
if DOCKER and BASE_READY:
    exec_result, shell_result = docker_stop_timing("stop-exec", EXEC_FORM), docker_stop_timing("stop-shell", SHELL_FORM)
    print("exec-форма: ", exec_result)
    print("shell-форма:", shell_result)
    assert exec_result["код"] == 0 and shell_result["код"] == 137
else:
    print("⏭ пропущено: немає Docker")

exec-форма: PID 1 — `python`, зупинка за долю секунди, код 0, у журналі «закриваю з'єднання». shell-форма: PID 1 — `/bin/sh -c …`; оболонка в ролі PID 1 `SIGTERM` не передає — Docker чекає всі 3 секунди й убиває (`137`), рядка про закриття немає.

### Приклад 8.6. Користувач `app`: код читати можна, змінювати — ні

Повторимо інструкції користувача з `Dockerfile` проєкту на маленькому образі й перевіримо права зсередини контейнера.

In [ ]:
USER_DEMO = """
FROM python:3.12-slim
WORKDIR /app
RUN useradd --uid 10001 --no-create-home --shell /usr/sbin/nologin app && mkdir /data && chown app /data
COPY app.py .
USER app
"""
USER_CHECKS = "id; echo '# зламано' >> /app/app.py && echo 'код змінено' || echo 'код змінити не можна'; " \
              "touch /data/news_hub.db && echo 'у /data писати можна'"

In [ ]:
if DOCKER and BASE_READY:
    (LAB / "user_context").mkdir(exist_ok=True)
    (LAB / "user_context" / "app.py").write_text("print('news_hub')\n")
    build("user", LAB / "user_context", USER_DEMO)
    output = docker("run", "--rm", PREFIX + "user", "sh", "-c", USER_CHECKS).stdout
    print(output)
    assert "uid=10001(app)" in output and "змінити не можна" in output
else:
    print("⏭ пропущено: немає Docker")

Процес працює від `uid=10001(app)`. Код скопійовано **до** `USER app`, тож він належить root — змінити його не можна навіть зламаному застосунку. Писати можна лише в `/data`, де лежить база (і куди підключають том).

### Приклад 8.7. Образ `news_hub`: збірка, `/health`, зупинка

Тепер — справжній `Dockerfile` проєкту. Збірка ставить залежності з PyPI, тож потрібна мережа (перша збірка — хвилина-дві). Запускаємо без Redis і без `.env`: `/health` відповідає, бо не торкається ні бази, ні Redis.

In [ ]:
def run_news_hub() -> dict[str, object]:
    started = time.perf_counter()
    docker("build", "--label", LABEL, "-t", PREFIX + "news-hub", str(PROJECT), timeout=900)
    docker_build_seconds = round(time.perf_counter() - started, 1)
    port, name = free_port(), PREFIX + "api"
    docker("rm", "-f", name, check=False)
    docker("run", "-d", "--name", name, "-p", f"127.0.0.1:{port}:8000", PREFIX + "news-hub")
    health = None
    for _ in range(60):
        try:
            with urllib.request.urlopen(f"http://127.0.0.1:{port}/health", timeout=2) as response:
                health = response.read().decode()
                break
        except OSError:
            time.sleep(1)
    user = docker("exec", name, "id", "-un").stdout.strip()
    in_app = docker("exec", name, "ls", "-A", "/app").stdout.split()
    since = docker("info", "--format", "{{.SystemTime}}").stdout.strip()
    started = time.perf_counter()
    docker("stop", name)
    seconds = round(time.perf_counter() - started, 1)
    code = docker("inspect", "-f", "{{.State.ExitCode}}", name).stdout.strip()
    tail = [line for line in docker("logs", name).stderr.splitlines()
            if "shutdown" in line.lower() or "Finished" in line]
    events = docker("events", "--since", since, "--until", "0s", "--filter", f"container={name}",
                    "--filter", "event=kill", "--format", "{{.Actor.Attributes.signal}}").stdout.split()
    docker("rm", name)
    size = docker("images", "--format", "{{.Size}}", PREFIX + "news-hub").stdout.strip()
    return {"збірка, с": docker_build_seconds, "образ": size, "/health": health, "користувач": user,
            "/app": in_app, "docker stop, с": seconds, "сигнали": events, "журнал": tail, "код": int(code)}

In [ ]:
if DOCKER and BASE_READY:
    try:
        result = run_news_hub()
        for key, value in result.items():
            print(f"{key:<15} {value}")
        assert result["/health"] and ".env" not in result["/app"]
        assert "INFO:     Application shutdown complete." in result["журнал"] and result["docker stop, с"] < 8
    except (RuntimeError, subprocess.TimeoutExpired) as error:
        print("⏭ образ news_hub не зібрано (найчастіше — немає доступу до PyPI):", str(error)[-300:])
else:
    print("⏭ пропущено: немає Docker")

Образ `news_hub` відповідає на `/health`, працює від `app`, у `/app` — лише код, міграції, дані й `requirements.txt` (без `.env`, тестів і `.venv`). `docker stop` займає 1–2 с, а не 10: `SIGTERM` дійшов до uvicorn, lifespan виконався (`Application shutdown complete.`), процес завершився (`Finished server process [1]`).

Рядок «сигнали» — з `docker events`: зазвичай там лише `15` (`SIGTERM`). Але буває, що Docker 29 через секунду-дві ще надсилає `SIGKILL` (`['15', '9']`) — саме тоді, коли Python уже завершується після lifespan. Тоді `docker inspect` покаже код `137` замість `0`, хоча журнал і час зупинки кажуть, що зупинка коректна. Тому про коректну зупинку судимо за журналом і часом, а код `137` **після 10 секунд** очікування — справжня ознака, що `SIGTERM` не дійшов. Повна перевірка того самого — `pytest -m docker` у папці проєкту (6 тестів).

### Приклад 8.8. Прибрати за собою

Видаляємо контейнери й образи з нашим префіксом і проміжні образи з нашою міткою. Чужих контейнерів і образів не торкаємося.

In [ ]:
def cleanup_docker() -> list[str]:
    removed = []
    for container in docker("ps", "-aq", "--filter", f"name={PREFIX}").stdout.split():
        docker("rm", "-f", container, check=False)
        removed.append(f"контейнер {container}")
    for image in docker("images", "--format", "{{.Repository}}:{{.Tag}}").stdout.split():
        if image.startswith(PREFIX):
            docker("rmi", "-f", image, check=False)
            removed.append(image)
    docker("image", "prune", "-f", "--filter", f"label={LABEL}", check=False)
    return removed

In [ ]:
if DOCKER:
    print("видалено:", cleanup_docker())
else:
    print("⏭ нічого прибирати: Docker не використовувався")
shutil.rmtree(LAB, ignore_errors=True)
print("docker_lab/ видалено")

### Вивід справжнього запуску (Docker 29, Linux)

Для тих, хто працює без Docker, — що надрукували клітинки розділу 8 на машині з Docker:

```text
8.2  зібрано за 1.1 с; файлів в образі: 35, за моделлю Python: 35
     однакові списки: True
8.3  правильний порядок: друга збірка 1.2 с, «pip» з кешу: True
     COPY . . спочатку:  друга збірка 4.8 с, «pip» з кешу: False
8.4  docker inspect Env: [PATH=/usr/local/bin:… LANG=C.UTF-8 … PYTHON_VERSION=3.12.15 … GEMINI_API_KEY=AIza-FAKE-KEY-FOR-LESSON]
     ls /app у контейнері: []
     ключ знайдено в шарі docker save: ['app/.env']
8.5  exec-форма:  {'PID 1': 'python /app/stop_app.py', 'секунд': 0.3, 'код': 0, 'журнал': ['сервер працює', "SIGTERM: закриваю з'єднання"]}
     shell-форма: {'PID 1': '/bin/sh -c python /app/stop_app.py', 'секунд': 3.3, 'код': 137, 'журнал': ['сервер працює']}
8.6  uid=10001(app) gid=10001(app) groups=10001(app)
     код змінити не можна
     у /data писати можна
8.7  збірка, с       34.5            (перша збірка: pip install з PyPI)
     образ           444MB
     /health         {"status":"ok"}
     користувач      app
     /app            ['alembic.ini', 'data', 'migrations', 'news_hub', 'requirements.txt']
     docker stop, с  1.4
     сигнали         ['15']
     журнал          ['INFO:     Waiting for application shutdown.', 'INFO:     Application shutdown complete.', 'INFO:     Finished server process [1]']
     код             0
```

---
## Локально, з Docker

```bash
cd module_5/lessons/lesson_49_docker/news_hub
docker build -t news_hub:48 .
docker network create news
docker run -d --name redis --network news redis:7-alpine
docker run --rm -v news_data:/data news_hub:48 alembic upgrade head
docker run -d --name api --network news -p 8000:8000 -v news_data:/data \
    -e REDIS_URL=redis://redis:6379/0 --env-file .env news_hub:48
curl localhost:8000/health/ready
docker stop api
pytest -m docker            # 6 тестів образу
```

```text
{"status":"ok","checks":{"database":"ok","redis":"ok"}}
```

## Самоперевірка

1. Чому `.env` не можна прибрати з образу командою `RUN rm .env`?
2. Змінився `requirements.txt`. Які інструкції виконаються заново?
3. Що означає `localhost` усередині контейнера?
4. Чим `/health` відрізняється від `/health/ready`?
5. Що означає код виходу 137?
6. Чому шаблон `*.db` у `.dockerignore` не виключає `data/news_hub.db`?

## Далі

- Книга курсу: [Урок 49](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m5/lesson_49/) — старий Dockerfile проти нового з реальними числами, томи, мережа, здоров'я і зупинка, архітектура.
- У папці `news_hub`: `pytest -m docker`.
- Урок 50 — Compose: API, міграції, PostgreSQL, Redis, nginx і бот однією командою; деплой.